# Activity 4: Night-time lights, and `config.py`

**The satellite.** NASA's Black Marble product (VIIRS, product `VNP46A2`) measures
artificial light at night at ~500 m resolution, every night, since January 2012,
everywhere on Earth. It arrives with a lag of about a week, months before GDP.

**The configuration file.** This is the first activity that reads `config.py`. The rule:

> **You edit `config.py`. You do not edit the notebook.**

The notebook has three parts:

1. **Download and extract** two weeks of tiles yourself: the whole pipeline at a small scale.
2. **The long view**: the full 2012–2026 national series, which we ship in
   `data/ntl_shape.csv` (extracting 14 years takes days and ~130 GB of tiles).
3. **Oil and gas**: what the lights over Pointe-à-Pierre, Point Lisas and Point Fortin
   tell us, and what they don't.

---

### Before you start: the `geo` environment

This notebook needs GDAL, HDF5 and geospatial libraries that are not in the `nwcst`
environment. **Run it with the `geo` kernel.** If the first cell says the HDF5 driver
is missing, see the install guide (`conda install -n geo -c conda-forge libgdal-hdf5`).

## The config loader

This cell is the same in every notebook that uses `config.py`. It finds `config.py`,
loads it as `cfg` (so every setting is `cfg.COUNTRY`, `cfg.TILES`, ...), derives the
standard folders into `P`, and imports the heavy lifting from `ntl_helpers.py`.
Read the banner it prints: everything in it came from `config.py`.

In [ ]:
# ===========================================================================
# CONFIG LOADER: edit config.py, NOT this cell.
# Gives you `cfg` (every setting) and `P` (the standard folders).
# ===========================================================================
import os, sys, importlib, importlib.util

# config.py lives in activity/. From activity/ that is the current folder;
# from sols/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))
_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)      # public/raw, public/data, public/output, ...

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import ntl_helpers
importlib.reload(ntl_helpers)      # pick up edits without restarting the kernel
from ntl_helpers import *

os.makedirs(P["nl_root"], exist_ok=True)   # public/hfiles/: the tiles you download

# Without GDAL's HDF5 driver nothing raises: every tile just reads as empty.
check_hdf5_driver()

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE}/{cfg.GADM_ISO3})")
print(f"         tiles      : {cfg.TILES}")
print(f"         window     : {cfg.NTL_DATE_RANGE}")
print(f"         max files  : {cfg.NTL_MAX_FILES}")
print(f"         tile store : {P['nl_root']}")
print(f"         outputs    : raw/{cfg.NTL_SHAPE_CSV}, raw/{cfg.NTL_OUTPUT_CSV}")

## The config file

Open `config.py` (same folder as this notebook). The settings that drive this notebook:

| setting | value for today | meaning |
|---|---|---|
| `TILES` | `["h11v07"]` | the 10°×10° satellite tile that contains Trinidad and Tobago |
| `NTL_DATE_RANGE` | two weeks | the nights to download and extract (~25 MB per night) |
| `NTL_MAX_FILES` | small | refuses a run above this many files, a guard against downloading years by mistake |
| `NTL_EVENT_DATE` | `None` | optional: a date to mark on the chart and centre the maps on |
| `NTL_DRAW_MAPS` | `True` | the before/after maps |

`NTL_DATE_RANGE = None` would mean *every night since 2012*: ~5,000 files and well
over 100 GB. Do not do that on a laptop.

## Your NASA token

The archive is free but not anonymous. You need an Earthdata account and a token:

1. Register at <https://urs.earthdata.nasa.gov>
2. Sign in at <https://ladsweb.modaps.eosdis.nasa.gov> → **My Account → Generate Token**
3. Paste the string into `NASA_TOKEN` in `config.py` and re-run the loader cell

Tokens expire after ~60 days; when downloads suddenly return `401`, that is why.
**Never share or publish a file that contains your token.**

In [ ]:
api_key    = cfg.NASA_TOKEN
HAVE_TOKEN = bool(api_key) and not api_key.startswith("PASTE")

inventory = {t: len(list_tile_rasters(P["nl_root"], [t], make=True)) for t in cfg.TILES}
print("Tiles on disk:")
for t, n in inventory.items():
    print(f"   {t}: {n:,} file(s)")
if not HAVE_TOKEN:
    print("\nNASA_TOKEN is empty in config.py -> the download will be skipped.")

## Download

`missing_targets` asks the NASA server which nights exist for the tile inside the
window, compares that with what is already on disk, and returns only the difference.
`download_many` fetches those files in parallel, with retries, behind the
`NTL_MAX_FILES` cap. Both live in `ntl_helpers.py`.

In [ ]:
from collections import Counter

start_date, end_date = cfg.ntl_window()
if end_date is not None:
    _days = (end_date - start_date).days + 1
    print(f"Window: {start_date.date()} -> {end_date.date()}  ({_days} nights, "
          f"~{_days * len(cfg.TILES) * 25} MB)")
else:
    print(f"Window: {start_date.date()} -> latest available (capped at {cfg.NTL_MAX_FILES} files)")

log_path = os.path.join(P["nl_root"], "ntl_download_log.txt")

if HAVE_TOKEN:
    target = missing_targets(cfg.download_tiles(), api_key, P["nl_root"],
                             start_date=start_date, end_date=end_date)
    print(f"{len(target)} file(s) on the server but not on disk.")
    if target:
        try:
            result = download_many(target, api_key, P["nl_root"], log_path=log_path,
                                   max_workers=4, max_files=cfg.NTL_MAX_FILES)
        except ntl_helpers.TooManyTargets as e:
            print("STOPPED --", e)      # deliberate: nothing was downloaded
            result = {}
        missing = result.get("unavailable", 0) + result.get("failed", 0)
        if result:
            print("All files downloaded." if missing == 0
                  else f"{missing} file(s) not downloaded; see {log_path}")
else:
    print("No token -> skipped.")

# Part 1: Measuring the country

## Step 1: the boundary

The national outline comes from **GADM**, the standard open database of administrative
boundaries. It is downloaded once and cached in `activity/shapes/`.

In [ ]:
import geopandas as gpd
import urllib.request

shape_dir = os.path.join(CODE_DIR, "shapes")
os.makedirs(shape_dir, exist_ok=True)
shape_zip = os.path.join(shape_dir, cfg.GADM_ZIP)
shape_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/" + cfg.GADM_ZIP

if not os.path.exists(shape_zip):
    print("Downloading boundary (once):", shape_url)
    urllib.request.urlretrieve(shape_url, shape_zip)

gdf         = gpd.read_file(f"zip://{shape_zip}!gadm41_{cfg.GADM_ISO3}_0.shp")
gdf_country = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")
print(f"bounds: {tuple(round(b, 2) for b in gdf_country.total_bounds)}")
gdf_country.explore(tiles="CartoDB dark_matter", zoom=9)

## Step 2: two questions, two functions

| Question | Function | What you give it |
|---|---|---|
| How lit is this **area**? | `processHD5_shape` | a boundary polygon |
| How lit is this **site**? | `processHD5` | a list of latitude/longitude points |

First the area: mean and total radiance over the national polygon, one row per night,
written to `raw/ntl_shape.csv`. Nights already in the file are skipped, so re-running
costs nothing.

In [ ]:
_ws, _we = cfg.ntl_window()
shape_csv = os.path.join(P["raw"], cfg.NTL_SHAPE_CSV)
files = rasters_in_range(P["nl_root"], list(cfg.TILES), start=_ws, end=_we)

done    = shape_dates_done(shape_csv, len(cfg.TILES), out_cols=cfg.NTL_SHAPE_COLS)
pending = pending_rasters(files, done, label="Country NTL")

recs = []
for _path, _dt in tqdm(pending, desc="Country NTL"):
    try:
        recs.append(processHD5_shape(_path, 2, P["ntl_tmp"], gdf_country, _dt))
    except Exception as e:
        print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

shape_df = update_shape_csv(recs, shape_csv, len(cfg.TILES), out_cols=cfg.NTL_SHAPE_COLS)
if not len(shape_df):
    raise RuntimeError("No data: hfiles/h11v07/ has nothing in this window. Did the download run?")
print(f"{len(shape_df)} night(s) in raw/{cfg.NTL_SHAPE_CSV}, "
      f"{shape_df.index.min().date()} to {shape_df.index.max().date()}")
shape_df[["ntl_mean", "ntl_sum"]]

In [ ]:
marks = ([(pd.Timestamp(cfg.NTL_EVENT_DATE), cfg.NTL_EVENT_LABEL or "event")]
         if cfg.NTL_EVENT_DATE else [])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(shape_df.index, shape_df["ntl_mean"], color="#005BAC", lw=1.6, marker="o", ms=3)
for _d, _lab in marks:
    ax.axvline(_d, color="#CC0000", lw=1.5, ls="--")
    ax.text(_d, ax.get_ylim()[1], f" {_lab}", color="#CC0000", fontsize=9, va="top")
ax.set_title(f"{cfg.COUNTRY} - nighttime lights, country mean, your window",
             fontsize=13, weight="bold")
ax.set_ylabel("nW/cm²/sr")
ax.grid(axis="y", ls=":", alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
print("saved ->", save_fig(fig, P, "ntl_country_mean", notebook="act4"))
plt.show()

**Why does the line jump around so much from night to night?** Clouds (a cloudy night
reads dark or is gap-filled), the moon, and the angle of the satellite. A single night
is noisy; the signal is in monthly averages and in long comparisons (Part 2).

## Step 3: before and after, on a map

The first and last night of your window (or `event ± NTL_EVENT_WINDOW` days if you set
an event in `config.py`), drawn as rasters.

In [ ]:
import contextily as cx
import colorcet as cc

if cfg.NTL_EVENT_DATE:
    _ev = pd.Timestamp(cfg.NTL_EVENT_DATE)
    d_before = _ev - pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    d_after  = _ev + pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    lab_before, lab_after = "before", "after"
else:
    d_before, d_after = shape_df.index.min(), shape_df.index.max()
    lab_before, lab_after = "window start", "window end"

f_before = find_h5s_for_date(d_before, P["nl_root"], list(cfg.TILES)) if cfg.NTL_DRAW_MAPS else []
f_after  = find_h5s_for_date(d_after,  P["nl_root"], list(cfg.TILES)) if cfg.NTL_DRAW_MAPS else []

if not (f_before and f_after):
    print("Maps skipped (NTL_DRAW_MAPS is False, or a tile is missing for those dates).")
else:
    arr_b, lons, lats = raster_to_array(f_before, 2, gdf_country, P["ntl_tmp"])
    arr_a, _, _       = raster_to_array(f_after,  2, gdf_country, P["ntl_tmp"])
    both = np.concatenate([arr_b[~np.isnan(arr_b)], arr_a[~np.isnan(arr_a)]])
    vmin, vmax = np.nanpercentile(both, 2), np.nanpercentile(both, 98)
    dx, dy = abs(lons[1] - lons[0]), abs(lats[1] - lats[0])
    extent = [lons[0] - dx / 2, lons[-1] + dx / 2, lats[-1] - dy / 2, lats[0] + dy / 2]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.subplots_adjust(bottom=0.15)
    for ax, arr, lab, d in zip(axes, [arr_b, arr_a], [lab_before, lab_after], [d_before, d_after]):
        im = ax.imshow(arr, cmap=cc.cm.bmy, vmin=vmin, vmax=vmax, extent=extent,
                       origin="upper", interpolation="nearest", zorder=2)
        gdf_country.boundary.plot(ax=ax, color="#444444", lw=0.8, zorder=3)
        ax.set_title(f"{lab}\n({d:%b %d, %Y})", fontsize=13, weight="bold")
        try:
            cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas,
                           crs="EPSG:4326", attribution_size=6, zorder=0)
        except Exception as e:
            print(f"  basemap unavailable ({type(e).__name__}), plotting without it")
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.set_aspect("equal")
    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    fig.colorbar(im, cax=cbar_ax, orientation="horizontal", label="radiance (nW/cm²/sr)")
    fig.suptitle(f"Nighttime light radiance - {cfg.COUNTRY}", fontsize=15, weight="bold")
    print("saved ->", save_fig(fig, P, "ntl_map_before_after", notebook="act4"))
    plt.show()

## Step 4: individual sites

The country mean is a blunt instrument. A refinery, a port, a hotel strip or a mall
each have their own series, and those connect to the sectors you are trying to nowcast:
energy, trade, tourism, retail.

`raw/Coordinates.xlsx` lists 76 sites in Trinidad and Tobago (sector, name, city,
latitude, longitude): 25 oil & gas, 20 hotels, 20 shopping malls, 6 seaports,
3 airports, 2 supermarkets. `processHD5` returns two readings per site per night:

* `DNBvalue1`: the single pixel containing the point;
* `DNBvalue3`: the mean of the 3×3 block around it. **Prefer this one**: a pixel is
  ~464 m across and geolocation is not perfect, so one pixel can miss a site by a cell.

Sites closer together than ~500 m share pixels: Point Lisas has eight sites inside a
couple of kilometres; they are not independent series.

In [ ]:
coord_path = os.path.join(P["raw"], cfg.COORDINATES_XLSX)
site_csv   = os.path.join(P["raw"], cfg.NTL_OUTPUT_CSV)

locations = pd.read_excel(coord_path)
for col in ("location", "city", "sector"):      # stable merge keys: no accents/spaces
    locations[col] = (locations[col].fillna("").astype(str).str.normalize("NFKD")
                        .str.encode("ascii", errors="ignore").str.decode("utf-8").str.strip())
coords = locations[["sector", "location", "city", "lat", "lon"]].to_dict("records")
print(f"{len(coords)} sites; sectors: " + ", ".join(sorted(locations["sector"].unique())))

done    = site_dates_done(site_csv, locations["location"])
pending = pending_rasters(files, done, label="Site NTL")

rows = []
for _path, _dt in tqdm(pending, desc="Site NTL"):
    try:
        rows.extend(processHD5(_path, 2, P["ntl_tmp"], coords, _dt))
    except Exception as e:
        print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

site_df = update_site_csv(rows, site_csv)
by_sector = site_df.groupby(["date", "sector"])["DNBvalue3"].mean().unstack("sector")
ax = by_sector.plot(figsize=(12, 4), lw=1.6, marker="o", ms=3, logy=True)
ax.set_title(f"{cfg.COUNTRY} - mean site radiance by sector (log scale)", fontsize=13, weight="bold")
ax.set_ylabel("nW/cm²/sr"); ax.set_xlabel("")
ax.legend(frameon=False, ncol=3, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Part 2: The long view

You just processed two weeks. The research pipeline has processed every night since
January 2012 the same way; the result ships as `data/ntl_shape.csv` (national) and
`data/ntl_sites_monthly.csv` (monthly median per site). From here on the notebook
reads only those files: no tiles, no token.

In [ ]:
hist = pd.read_csv(os.path.join(P["data"], "ntl_shape.csv"), parse_dates=["date"]).set_index("date")
print(f"{len(hist):,} nights, {hist.index.min().date()} to {hist.index.max().date()}")

monthly = hist["shape_mean"].resample("MS").median()      # median: robust to cloudy nights

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hist.index, hist["shape_mean"], color="#9bbbd8", lw=0.4, label="daily")
ax.plot(monthly.index, monthly, color="#005BAC", lw=1.8, label="monthly median")
ax.set_ylim(0, hist["shape_mean"].quantile(0.995))
ax.set_title(f"{cfg.COUNTRY} - nighttime lights, country mean, 2012-2026", fontsize=13, weight="bold")
ax.set_ylabel("nW/cm²/sr"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Part 3: Oil, gas and night-time lights

Energy plant is **bright, continuous and fixed in space**, and its start-ups and
shutdowns have **known dates**. That lets us test the satellite against something we
already know, a stricter test than "lights correlate with GDP".

Trinidad's natural experiment: **Petrotrin closed the Pointe-à-Pierre refinery in
November 2018** (it is now owned by Guaracara Refining). Compare:

| site | role in the test |
|---|---|
| Guaracara Refining (Pointe-à-Pierre refinery) | **treated**: shut down Nov 2018 |
| Point Lisas Industrial Estate | **comparison**: petrochemicals, kept operating |
| Atlantic LNG terminal (Point Fortin) | gas: LNG trains depend on gas supply |

In [ ]:
sites = pd.read_csv(os.path.join(P["data"], "ntl_sites_monthly.csv"), parse_dates=["date"])
sites

In [ ]:
og = sites[sites["sector"] == "Oil & Gas"]
print("Oil & gas sites:\n  " + "\n  ".join(sorted(og["location"].unique())))

pick = {"Pointe-a-Pierre refinery":      "Guaracara",
        "Point Lisas Industrial Estate": "Point Lisas Industrial Estate",
        "Atlantic LNG (Point Fortin)":   "Atlantic LNG"}
wide = pd.DataFrame({lab: og[og["location"].str.contains(key)].set_index("date")["ntl_median"]
                     for lab, key in pick.items()})

ax = wide.rolling(3, min_periods=2).mean().plot(figsize=(12, 4.5), lw=1.6, logy=True)
ax.axvline(pd.Timestamp("2018-11-30"), color="#CC0000", ls="--", lw=1.2)
ax.text(pd.Timestamp("2018-12-15"), ax.get_ylim()[1] * 0.7, " refinery closes",
        color="#CC0000", fontsize=9)
ax.set_title("Monthly median radiance at oil & gas sites (3-month mean, log scale)",
             fontsize=13, weight="bold")
ax.set_ylabel("nW/cm²/sr"); ax.set_xlabel("")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()